In [1]:
!pip install tensorflow==2.15.0 keras==2.15.0 numpy==1.24.3 pandas==1.5.3 matplotlib==3.7.1 scikit-learn==1.3.0 protobuf==3.20.3

  Using cached numpy-1.24.3-cp310-cp310-win_amd64.whl (14.8 MB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Your Docs\\venv\\Lib\\site-packages\\~umpy.libs\\libopenblas64__v0.3.23-293-gc2f4bdbb-gcc_10_3_0-2bde3a66a51006b2b53eb373ff767a3f.dll'
Check the permissions.

You should consider upgrading via the 'C:\Your Docs\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

try:
	tokens
except NameError:
	with open("tokens.pkl", "rb") as f:
		tokens = pickle.load(f)

tokenizer = Tokenizer()
tokenizer.fit_on_texts([" ".join(tokens)])

vocab_size = len(tokenizer.word_index) + 1
print("Vocab size:", vocab_size)

ids = tokenizer.texts_to_sequences([" ".join(tokens)])[0]
print("Sample ids:", ids[:10])


Vocab size: 31528
Sample ids: [2069, 852, 5, 171, 1431, 7, 72, 132, 13765, 2]


In [4]:
import sys
!{sys.executable} -m pip install gensim

from gensim.models import Word2Vec
import numpy as np

# Word2Vec train karo
w2v = Word2Vec(sentences=[tokens], vector_size=64, window=5, min_count=1, workers=4)

# Embedding matrix banao — har word ka vector nikalo
embedding_matrix = np.zeros((vocab_size, 64))

for word, idx in tokenizer.word_index.items():
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]

print("Embedding matrix shape:", embedding_matrix.shape)

'c:\Your' is not recognized as an internal or external command,
operable program or batch file.


Embedding matrix shape: (31528, 64)


In [5]:
import numpy as np

SEQ_LENGTH = 3
X, y = [], []

for i in range(SEQ_LENGTH, len(ids)):
    X.append(ids[i - SEQ_LENGTH : i])
    y.append(ids[i])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1362897, 3)
y shape: (1362897,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)

Train: (1090317, 3)
Test:  (272580, 3)


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 64, weights=[embedding_matrix], trainable=True),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 64)          2017792   
                                                                 
 lstm (LSTM)                 (None, None, 128)         98816     
                                                                 
 dropout (Dropout)           (None, None, 128)         0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 31528)             2049320   
                                                                 
Total params: 4215336 (16.08 MB)
Trainable params: 421

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=3, restore_best_weights=True)

model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5


17037/17037 [==============================] - 636s 37ms/step - loss: 6.6416 - accuracy: 0.1219 - val_loss: 6.2499 - val_accuracy: 0.1540
Epoch 2/5
17037/17037 [==============================] - 695s 41ms/step - loss: 6.1592 - accuracy: 0.1577 - val_loss: 6.0618 - val_accuracy: 0.1667
Epoch 3/5
17037/17037 [==============================] - 852s 50ms/step - loss: 5.9874 - accuracy: 0.1677 - val_loss: 5.9729 - val_accuracy: 0.1735
Epoch 4/5
17037/17037 [==============================] - 902s 53ms/step - loss: 5.8759 - accuracy: 0.1745 - val_loss: 5.9183 - val_accuracy: 0.1771
Epoch 5/5
17037/17037 [==============================] - 772s 45ms/step - loss: 5.7946 - accuracy: 0.1803 - val_loss: 5.8782 - val_accuracy: 0.1816


In [10]:
import pickle

# Model save karo
model.save("lstm_model.keras")
model.save("lstm_model.h5")
model.save("lstm_tokenizer.pkl")

# # Tokenizer bhi save karo — ye zaroori hai predict ke liye
# with open("lstm_tokenizer.pkl", "wb") as f:
#     pickle.dump(tokenizer, f)

print("Saved → lstm_model.h5 ✅")
print("Saved → lstm_model.keras ✅")

c:\Your Docs\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: lstm_tokenizer.pkl\assets


INFO:tensorflow:Assets written to: lstm_tokenizer.pkl\assets


Saved → lstm_model.h5 ✅
Saved → lstm_model.keras ✅


In [ ]:
import pickle

with open("lstm_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

: 